In [1]:
%run 0_1_load_paths.ipynb

In [2]:
import dataclasses
import collections
import os

import minervapy

import momapy.core
import momapy.celldesigner.io.celldesigner
import momapy.celldesigner.io.pickle
import momapy.io

import momapy_kb.neo4j.core

import commute_dm.utils

import credentials

In [3]:
def get_maps_to_dir(url, map_dir, format_, project_id=None, verbose=False, log_in=True):
    minervapy.set_base_url(url)
    if log_in:
        minervapy.log_in(credentials.MINERVA_USERNAME, credentials.MINERVA_PASSWORD)
    if project_id is None:
        config = minervapy.get_configuration()
        project_id = config.get_option("DEFAULT_MAP").value
    project = minervapy.get_project(project_id)
    maps = minervapy.get_maps(project)
    extensions = {"celldesigner": "xml", "sbgnml": "sbgn"}
    extension = extensions[format_]
    for map_ in maps:
        if map_.name != "overview":
            output_file_path = os.path.join(
                map_dir, f"{map_.name.replace(' ', '_')}.{extension}"
            )
            if verbose:
                print(f"Downloading '{map_.name}' under format '{format_}'...")
            try:
               map_.download(format_=format_, output_file_path=output_file_path)
            except minervapy.utils.StatusCodeException as e:
                print(f"Could not download '{map_.name}' under format '{format_}': '{e}', skipping")

## Getting the maps from MINERVA

In [4]:
commute_dm.utils.remake_dir(COVID_DM_CD_BUILD_DIR)
commute_dm.utils.remake_dir(PD_DM_CD_BUILD_DIR)

In [5]:
COVID_URL = "https://covid19map.elixir-luxembourg.org/minerva/api/"
PD_URL = "https://pdmap.uni.lu/minerva/api/"
PD_PROJECT_ID = "pd_map_summer_25"
FORMAT = "celldesigner"

We download the maps from MINERVA:

In [6]:
get_maps_to_dir(COVID_URL, COVID_DM_CD_BUILD_DIR, format_=FORMAT)

In [7]:
get_maps_to_dir(PD_URL, PD_DM_CD_BUILD_DIR, format_=FORMAT, project_id=PD_PROJECT_ID, log_in=True)

## Making pickle versions

In [8]:
commute_dm.utils.remake_dir(COVID_DM_CD_PICKLE_BUILD_DIR)
commute_dm.utils.remake_dir(PD_DM_CD_PICKLE_BUILD_DIR)

In [9]:
for input_dir_path, output_dir_path in [
    (COVID_DM_CD_BUILD_DIR, COVID_DM_CD_PICKLE_BUILD_DIR),
    (PD_DM_CD_BUILD_DIR, PD_DM_CD_PICKLE_BUILD_DIR)
]:
    for input_file_name, input_file_path in commute_dm.utils.list_dir(input_dir_path):
        output_file_name = commute_dm.utils.rename_file(input_file_name, "pickle")
        output_file_path = os.path.join(output_dir_path, output_file_name)
        reader_result = momapy.io.read(input_file_path)
        writer_result = momapy.io.write(
            obj=reader_result.obj,
            file_path=output_file_path,
            writer="celldesigner_pickle",
            annotations=reader_result.annotations,
            notes=reader_result.notes,
            ids=reader_result.ids
        )